In [1]:
import pandas as pd
import pickle
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [2]:
df = pd.read_csv("clean_ntpc_data.csv")
df.head()

,equipment,Functional Loc.,Priority,Notifictn type,month,day_of_week,hour,shift,year,days_to_next_defect
0,1,446,1,0,1,5,11,1,2023,52
1,2,447,1,1,3,1,10,1,2023,21
2,2,447,1,1,4,1,11,1,2023,157
3,2,447,1,1,9,4,16,0,2023,13
4,2,447,1,1,9,3,10,1,2023,30


In [3]:
# X = input features (what we give the model)
# y = target (what we want to predict)

x = df.drop(columns=['days_to_next_defect'])
y = df['days_to_next_defect']

print("X shape:", x.shape)
print("y shape:", y.shape)

X shape: (6846, 9)
y shape: (6846,)


In [4]:
# Converting days to risk levels
# More useful for plant engineers than exact days

def get_risk(days):
    if days <= 30:
        return 'HIGH'      # defect coming soon
    elif days <= 90:
        return 'MEDIUM'    # defect coming in medium term
    else:
        return 'LOW'       # defect not imminent

y = y.apply(get_risk)

print("Risk distribution:")
print(y.value_counts())

Risk distribution:
days_to_next_defect
HIGH      3185
LOW       1960
MEDIUM    1701
Name: count, dtype: int64


In [5]:
# 80% for training 20% for testing
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

print("Training rows:", x_train.shape[0])
print("Testing rows :", x_test.shape[0])

Training rows: 5476
Testing rows : 1370


In [6]:
# Training Random Forest Classifier
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(x_train, y_train)

print("Model trained successfully!")

Model trained successfully!


In [7]:
predictions = model.predict(x_test)
accuracy = accuracy_score(y_test, predictions) * 100

print("Accuracy:", round(accuracy, 2), "%")
print("\nSample predictions vs actual:")
for i in range(5):
    print(f"Predicted: {predictions[i]:6}  Actual: {y_test.values[i]}")

Accuracy: 50.51 %

Sample predictions vs actual:
Predicted: HIGH    Actual: MEDIUM
Predicted: HIGH    Actual: MEDIUM
Predicted: HIGH    Actual: MEDIUM
Predicted: LOW     Actual: MEDIUM
Predicted: HIGH    Actual: HIGH


In [9]:
# Loading original clean data to calculate averages
df_orig = pd.read_csv("clean_ntpc_data.csv")

# Adding risk column
df_orig['risk'] = df_orig['days_to_next_defect'].apply(get_risk)

# Calculating average days per equipment per risk level
avg_days = df_orig.groupby(['equipment', 'risk'])['days_to_next_defect'].mean().round().astype(int)

# Convert to dictionary for easy lookup in app
avg_days_dict = {}
for (equipment, risk), days in avg_days.items():
    if equipment not in avg_days_dict:
        avg_days_dict[equipment] = {}
    avg_days_dict[equipment][risk] = days

print("Average days calculated!")
print("\nSample — Equipment 233:")
print(avg_days_dict.get(233, "Not found"))

Average days calculated!

Sample — Equipment 233:
{'HIGH': 3, 'LOW': 123, 'MEDIUM': 40}


In [10]:
# Saving model
pickle.dump(model, open('model.pkl', 'wb'))

# Saving average days dictionary
pickle.dump(avg_days_dict, open('avg_days.pkl', 'wb'))

# Saving feature columns
feature_columns = list(x.columns)
pickle.dump(feature_columns, open('feature_columns.pkl', 'wb'))

print("All files saved!")
print("\nFiles created:")
print("→ model.pkl           (trained RF classifier)")
print("→ avg_days.pkl        (average days per equipment per risk)")
print("→ feature_columns.pkl (columns used for training)")
print("→ label_encoders.pkl  (already saved in step2)")
print("→ clean_ntpc_data.csv (already saved in step2)")
print("\nStep 3 Complete! ✅")

All files saved!

Files created:
→ model.pkl           (trained RF classifier)
→ avg_days.pkl        (average days per equipment per risk)
→ feature_columns.pkl (columns used for training)
→ label_encoders.pkl  (already saved in step2)
→ clean_ntpc_data.csv (already saved in step2)

Step 3 Complete! ✅


In [11]:
import pickle

pickle.dump(model, open('model.pkl', 'wb'))
pickle.dump(avg_days_dict, open('avg_days.pkl', 'wb'))

print("model.pkl saved!")
print("avg_days.pkl saved!")

model.pkl saved!
avg_days.pkl saved!
